In [1]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

/kaggle/input/datasets/chitreshkr/idnet-identity-document-analysis/files-archive


In [2]:
base = "/kaggle/input/datasets/chitreshkr/idnet-identity-document-analysis"
for root, dirs, files in os.walk(base):
    print(root)
    for f in files:
        full_path = os.path.join(root, f)
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f"  {f} — {size_mb:.1f} MB")

/kaggle/input/datasets/chitreshkr/idnet-identity-document-analysis
  files-archive — 46647.0 MB


In [3]:
import subprocess
result = subprocess.run(["file", "/kaggle/input/datasets/chitreshkr/idnet-identity-document-analysis/files-archive"], capture_output=True, text=True)
print(result.stdout)

/kaggle/input/datasets/chitreshkr/idnet-identity-document-analysis/files-archive: Zip archive data, at least v4.5 to extract, compression method=store



In [4]:
import zipfile

archive_path = "/kaggle/input/datasets/chitreshkr/idnet-identity-document-analysis/files-archive"

with zipfile.ZipFile(archive_path, 'r') as z:
    names = z.namelist()
    print(f"Total entries: {len(names)}")
    for n in names:
        print(n)

Total entries: 3
GRC.zip
WV.zip
RUS.zip


In [5]:
import os

out_dir = "/kaggle/working/raw_sample"
os.makedirs(out_dir, exist_ok=True)

COUNTRY_TO_LABEL = {
    "WV.zip": "drivers_license",
    "RUS.zip": "national_id",
    "GRC.zip": "passport",
}

PER_CATEGORY = 200

with zipfile.ZipFile(archive_path, 'r') as outer_zip:
    for country_zip, label in COUNTRY_TO_LABEL.items():
        with outer_zip.open(country_zip) as inner_file:
            with zipfile.ZipFile(inner_file, 'r') as inner_zip:
                names = inner_zip.namelist()
                country_prefix = country_zip.replace(".zip", "")

                genuine_candidates = [
                    n for n in names
                    if n.startswith(f"{country_prefix}/positive/")
                    and n.lower().endswith((".jpg", ".png"))
                ]

                selected = genuine_candidates[:PER_CATEGORY]
                print(f"{label}: {len(selected)} images")

                for name in selected:
                    flat_name = f"{label}_" + os.path.basename(name)
                    target_path = os.path.join(out_dir, flat_name)
                    with inner_zip.open(name) as src, open(target_path, "wb") as dst:
                        dst.write(src.read())

print(f"\nDone. Total files: {len(os.listdir(out_dir))}")

drivers_license: 200 images
national_id: 200 images
passport: 200 images

Done. Total files: 600


In [6]:
import shutil
shutil.make_archive("/kaggle/working/raw_sample", 'zip', "/kaggle/working/raw_sample")

'/kaggle/working/raw_sample.zip'

In [7]:
from collections import Counter

files = os.listdir("/kaggle/working/raw_sample")
counts = Counter()
for f in files:
    for label in ["drivers_license", "national_id", "passport"]:
        if f.startswith(label):
            counts[label] += 1
for k, v in counts.items():
    print(f"{k}: {v}")
print(f"Total: {sum(counts.values())}")

national_id: 200
passport: 200
drivers_license: 200
Total: 600
